# Optimizers in Practice

从 SGD 到 Adam：动量、自适应学习率如何加速收敛。本课在同一个任务上横评四种优化器，并演示学习率调度、warmup 与梯度裁剪。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 四种优化器的更新规则


| 优化器 | 更新规则 | 关键思想 |
|--------|----------|----------|
| SGD | $w \gets w - \eta g$ | 基础 |
| Momentum | $v \gets \beta v + g;\ w \gets w - \eta v$ | 沿历史方向累积，穿越山谷 |
| RMSprop | $v \gets \beta v + (1-\beta)g^2;\ w \gets w - \frac{\eta}{\sqrt{v}+\epsilon}g$ | 按梯度幅度逐参数缩放 |
| Adam | 动量 + RMSprop + 偏差校正 | 两者的结合 |

直觉：动量看"方向"，RMSprop/Adam 看"每维的尺度"。


In [ ]:
rng = np.random.default_rng(42)
n = 400
X0 = rng.standard_normal((n, 2)) + np.array([-2.0, 0.0])
X1 = rng.standard_normal((n, 2)) + np.array([2.0, 0.0])
X = np.vstack([X0, X1]); y = np.concatenate([np.zeros(n), np.ones(n)])
Xt = torch.tensor(X, dtype=torch.float32); yt = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

def run_opt(make_opt, steps=400, bs=64, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = make_opt(model.parameters())
    losses = []
    for step in range(steps):
        opt.zero_grad()
        idx = torch.randint(0, len(Xt), (bs,))
        loss = F.binary_cross_entropy(torch.sigmoid(model(Xt[idx])), yt[idx])
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

curves = {
    'SGD lr=0.05':        run_opt(lambda p: torch.optim.SGD(p, lr=0.05)),
    'Momentum 0.9':       run_opt(lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
    'RMSprop lr=0.005':   run_opt(lambda p: torch.optim.RMSprop(p, lr=0.005)),
    'Adam lr=0.01':       run_opt(lambda p: torch.optim.Adam(p, lr=0.01)),
}
plt.figure(figsize=(9, 5))
for name, ls in curves.items():
    plt.plot(ls, label=name)
plt.xlabel('step'); plt.ylabel('BCE loss')
plt.title('同一网络、同一数据：优化器横评')
plt.legend(); plt.grid(alpha=0.3)


## 2. Adam 的超参数


- $\beta_1$（一阶动量，默认 0.9）：方向平滑
- $\beta_2$（二阶动量，默认 0.999）：尺度估计
- $\epsilon$（默认 1e-8）：防除零
- 偏差校正：前几步估计偏小，$\hat v_t = v_t/(1-\beta_2^t)$ 修正

**实践**：先试默认；loss 不稳就降 lr；$\epsilon$ 太小（1e-8）在低精度训练可能出 NaN，可用 1e-6~1e-4。


In [ ]:
# Adam 不同学习率
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for lr in [0.001, 0.01, 0.05]:
    axes[0].plot(run_opt(lambda p: torch.optim.Adam(p, lr=lr), steps=200), label=f'lr={lr}')
axes[0].set_title('Adam：学习率'); axes[0].set_xlabel('step'); axes[0].legend(); axes[0].grid(alpha=0.3)

for eps in [1e-8, 1e-4, 1e-2]:
    axes[1].plot(run_opt(lambda p: torch.optim.Adam(p, lr=0.01, eps=eps), steps=200), label=f'eps={eps}')
axes[1].set_title('Adam：ε'); axes[1].set_xlabel('step'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()


## 3. 学习率调度


训练中逐步降低学习率：先大步探索，后小步精调。

- **StepLR**：每 N 轮 ×γ
- **CosineAnnealing**：余弦从初始到 0，平滑下降


In [ ]:
def run_schedule(make_sched, steps=400, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    sched = make_sched(opt)
    losses, lrs = [], []
    for step in range(steps):
        opt.zero_grad()
        idx = torch.randint(0, len(Xt), (64,))
        loss = F.binary_cross_entropy(torch.sigmoid(model(Xt[idx])), yt[idx])
        loss.backward(); opt.step(); sched.step()
        losses.append(loss.item()); lrs.append(opt.param_groups[0]['lr'])
    return losses, lrs

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
l_const, _ = run_schedule(lambda opt: torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1.0))
l_step, lr_step = run_schedule(lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.5))
l_cos, lr_cos = run_schedule(lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=400))
axes[0].plot(l_const, label='constant'); axes[0].plot(l_step, label='StepLR'); axes[0].plot(l_cos, label='Cosine')
axes[0].set_xlabel('step'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('损失曲线')
axes[1].plot(lr_step, label='StepLR'); axes[1].plot(lr_cos, label='Cosine')
axes[1].set_xlabel('step'); axes[1].set_ylabel('学习率'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('学习率曲线')
plt.tight_layout()


## 4. Warmup 与梯度裁剪


- **Warmup**：前若干步学习率从 0 线性升到目标值——避免开局大梯度破坏预训练/大 batch 的统计
- **梯度裁剪**：$g \gets g \cdot \min(1, \frac{\text{clip}}{\|g\|})$——把梯度范数限制在阈值内，防爆炸（尤其 RNN/Transformer）


In [ ]:
def run_with_clip(clip, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    losses = []
    for step in range(400):
        opt.zero_grad()
        idx = torch.randint(0, len(Xt), (64,))
        loss = F.binary_cross_entropy(torch.sigmoid(model(Xt[idx])), yt[idx])
        loss.backward()
        if clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        losses.append(loss.item())
    return losses

plt.figure(figsize=(8, 4.5))
plt.plot(run_with_clip(None), label='无裁剪')
plt.plot(run_with_clip(1.0), label='裁剪到 1.0')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
plt.title('梯度裁剪：防爆炸（本任务温和，差异小；RNN/大模型场景关键）')


## 5. 实操清单


1. **默认起手**：Adam(lr=1e-3)，不行再调
2. **收敛不理想**：试 SGD+Momentum(0.9) + 余弦调度（某些任务泛化更好）
3. **大模型/Transformer**：AdamW + warmup + 梯度裁剪
4. **小数据小模型**：SGD 足够，省内存
5. **遇到 NaN**：降 lr、查 ε、查数据、查 loss 公式


## 课后练习


1. **动手实现**：不用 torch.optim，手写 Momentum 与 RMSprop 的更新步骤（对照公式）。
2. **超参扫描**：对 Adam 扫 lr ∈ {1e-4, 1e-3, 1e-2, 1e-1}，记录最优值。
3. **余弦 vs Step**：把 StepLR 的 γ 改成 0.9、step 改成 50，比较与余弦的差异。
4. **裁剪阈值**：把 clip 从 1.0 改成 0.1 与 10，观察损失曲线。
5. **思考**：为什么自适应优化器（Adam）在测试集上有时不如调好的 SGD+Momentum？
